## Setup & Imports

In [2]:
import os
import time
import pandas as pd
import numpy as np

from app import (
    detect_lang,
    translate,
    is_small_talk,
    answer_user_query,
    retriever,
)

print("Backend imported successfully ✔️")

c:\Users\Ashwin\anaconda3\envs\medibot_v2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Backend imported successfully ✔️


## Language Detection Evaluation

In [2]:
test_cases_lang = [
    ("Hi", "en"),
    ("How are you?", "en"),
    ("I have fever for 2 days", "en"),

    ("எனக்கு காய்ச்சல் இருக்கு", "ta"),
    ("மூச்சு திணறல் இருக்கிறது", "ta"),

    ("मुझे पेट दर्द है", "hi"),
    ("मुझे बुखार आ रहा है", "hi"),

    ("Tengo dolor de cabeza", "es"),
    ("Tengo tos desde ayer", "es"),

    ("لدي صداع شديد", "ar"),
    ("بطني يؤلمني", "ar"),

    ("頭が痛いです", "ja"),

    ("Hiiiiii", "en"),
    ("Hi bro what's up?", "en"),
]

records = []
for text, expected in test_cases_lang:
    detected = detect_lang(text)
    records.append({
        "Input": text,
        "Expected": expected,
        "Detected": detected,
        "Match": expected == detected
    })

df_lang = pd.DataFrame(records)
display(df_lang)

print("Language Detection Accuracy:", df_lang["Match"].mean() * 100, "%")

,Input,Expected,Detected,Match
0,Hi,en,en,True
1,How are you?,en,en,True
2,I have fever for 2 days,en,da,False
3,எனக்கு காய்ச்சல் இருக்கு,ta,ta,True
4,மூச்சு திணறல் இருக்கிறது,ta,ta,True
5,मुझे पेट दर्द है,hi,hi,True
6,मुझे बुखार आ रहा है,hi,hi,True
7,Tengo dolor de cabeza,es,es,True
8,Tengo tos desde ayer,es,es,True
9,لدي صداع شديد,ar,fa,False


Language Detection Accuracy: 85.71428571428571 %


## Translation Quality Evaluation

In [3]:
translation_tests = [
    ("ta", "எனக்கு வயிற்று வலி இருக்கு"),
    ("hi", "मुझे बुखार और कमजोरी है"),
    ("es", "Tengo tos desde ayer"),
    ("ar", "لدي التهاب في الحلق"),
    ("fr", "J'ai mal à la gorge"),
]

rows = []
for lang, text in translation_tests:
    to_en = translate(text, target="en", source=lang)
    back = translate(to_en, target=lang, source="en")
    
    rows.append({
        "Language": lang,
        "Original": text,
        "To English": to_en,
        "Back Translation": back,
    })

df_trans = pd.DataFrame(rows)
display(df_trans)

,Language,Original,To English,Back Translation
0,ta,எனக்கு வயிற்று வலி இருக்கு,I have a stomach ache,எனக்கு வயிறு வலிக்கிறது
1,hi,मुझे बुखार और कमजोरी है,I have fever and weakness,मुझे बुखार और कमजोरी है
2,es,Tengo tos desde ayer,I have a cough since yesterday,tengo tos desde ayer
3,ar,لدي التهاب في الحلق,I have a sore throat,لدي التهاب في الحلق
4,fr,J'ai mal à la gorge,I have a sore throat,j'ai mal à la gorge


## Small-Talk Detection Evaluation

In [4]:
small_talk = [
    ("Hi", True),
    ("Hello", True),
    ("Good morning", True),
    ("Thanks", True),
    ("Thank you", True),
    ("What's up", True)
]

not_small_talk = [
    ("I have chest pain", False),
    ("My stomach is hurting", False),
    ("I feel feverish", False),
    ("I feel dizzy", False),
    ("Severe cough since 2 days", False),
]

tests = small_talk + not_small_talk

rows = []
for text, expected in tests:
    pred = is_small_talk(text)
    rows.append({
        "Input": text,
        "Expected": expected,
        "Predicted": pred,
        "Correct": expected == pred
    })

df_small = pd.DataFrame(rows)
display(df_small)

print("Small-talk Detection Accuracy:", df_small["Correct"].mean() * 100, "%")

,Input,Expected,Predicted,Correct
0,Hi,True,True,True
1,Hello,True,True,True
2,Good morning,True,True,True
3,Thanks,True,True,True
4,Thank you,True,True,True
5,What's up,True,True,True
6,I have chest pain,False,False,True
7,My stomach is hurting,False,False,True
8,I feel feverish,False,False,True
9,I feel dizzy,False,False,True


Small-talk Detection Accuracy: 100.0 %


## RAG Retrieval Quality Evaluation

In [7]:
rag_questions = [
    "What are symptoms of dengue?",
    "How to treat fever?",
    "What causes stomach cramps?",
    "Symptoms of malaria",
    "What to do for chest pain?"
]

retrieval_scores = []

for q in rag_questions:
    print("=" * 70)
    print("QUESTION:", q)
    
    docs = retriever.invoke(q)
    
    for i, d in enumerate(docs):
        print(f"\n--- Document {i+1} ---")
        print(d.page_content[:400], "...")
        print("Source:", d.metadata.get("source"))
    
    score = int(input("\nRate relevance 0–5: "))
    retrieval_scores.append({"Question": q, "Score": score})

df_rag = pd.DataFrame(retrieval_scores)
display(df_rag)

print("Avg Retrieval Score:", df_rag["Score"].mean())

QUESTION: What are symptoms of dengue?

--- Document 1 ---
but the other initial symptoms of dengue fever are
absent. The patient develops acough, followed by the
appearance of small purplish spots (petechiae) on the
skin. These petechiae are areas where blood is leaking
out of the vessels. Large bruised areas appear as the
bleeding worsens and abdominal pain may be severe.
The patient may begin to vomit a substance that looks
like coffee grounds. This is ...
Source: Data\Data.pdf

--- Document 2 ---
2004 several cases were reported along the border
between Texas and Mexico in the southwestern
United States. This virus causes either the mild dengue
fever or the more serious dengue hemorrhagic fever–
dengue shock syndrome (DHF-DSS).
In children, dengue fever is characterized by a sore
throat, runny nose, slight cough, and a fever lasting for
a week or less. Older children and adults experience
 ...
Source: Data\Data.pdf

--- Document 3 ---
lasts one to seven days, after which the sympto

,Question,Score
0,What are symptoms of dengue?,4
1,How to treat fever?,5
2,What causes stomach cramps?,5
3,Symptoms of malaria,4
4,What to do for chest pain?,4


Avg Retrieval Score: 4.4


## Model Answer Quality Evaluation (Multilingual)

In [5]:
eval_set = [
    ("en", "What are the symptoms of dengue fever?"),
    ("ta", "டெங்குவின் அறிகுறிகள் என்ன?"),
    ("hi", "डेंगू के लक्षण क्या हैं?"),
    ("es", "¿Cuáles son los síntomas del dengue?"),
    ("ar", "ما هي أعراض حمى الضنك؟")
]

scores = []

for lang, q in eval_set:
    print("=" * 70)
    print(f"USER INPUT ({lang}):", q)
    
    ans, extras = answer_user_query(q)
    
    print("\nDetected:", extras["detected_lang_code"], extras["detected_lang_name"])
    print("Q English:", extras["question_en"])
    print("\nAnswer (User Language):", ans)
    print("\nAnswer English:", extras["answer_en"])
    
    acc  = int(input("\nAccuracy (1–5): "))
    rel  = int(input("Relevance (1–5): "))
    safe = int(input("Safety (1–5): "))
    hall = int(input("Hallucination (1–5, lower=better): "))
    
    scores.append({
        "Lang": lang,
        "Question": q,
        "Accuracy": acc,
        "Relevance": rel,
        "Safety": safe,
        "Hallucination": hall
    })

df_answers = pd.DataFrame(scores)
display(df_answers)

print("\nAVERAGES:")
print("Accuracy:", df_answers["Accuracy"].mean())
print("Relevance:", df_answers["Relevance"].mean())
print("Safety:", df_answers["Safety"].mean())
print("Hallucination:", df_answers["Hallucination"].mean())

USER INPUT (en): What are the symptoms of dengue fever?

Detected: en English
Q English: What are the symptoms of dengue fever?

Answer (User Language): Symptoms of dengue fever include:

- Fever
- Headache
- Muscle and joint pain
- Sore throat
- Runny nose
- Slight cough
- Small purplish spots (petechiae) on the skin
- Large bruised areas
- Abdominal pain
- Vomiting (may look like coffee grounds)
- Jaundice
- Delirium
- Seizures
- Stupor
- Coma
- Bleeding from mucous membranes and under the skin
- Dark blood in stools and vomit

In severe cases, it can lead to dengue hemorrhagic fever or dengue shock syndrome.

*Disclaimer: This information is for educational purposes only and should not replace professional medical advice, diagnosis, or treatment. Always consult a healthcare provider for any medical concerns.*

Answer English: Symptoms of dengue fever include:

- Fever
- Headache
- Muscle and joint pain
- Sore throat
- Runny nose
- Slight cough
- Small purplish spots (petechiae) on t

,Lang,Question,Accuracy,Relevance,Safety,Hallucination
0,en,What are the symptoms of dengue fever?,5,5,5,1
1,ta,டெங்குவின் அறிகுறிகள் என்ன?,4,4,5,1
2,hi,डेंगू के लक्षण क्या हैं?,4,4,5,1
3,es,¿Cuáles son los síntomas del dengue?,5,5,5,1
4,ar,ما هي أعراض حمى الضنك؟,4,4,5,1



AVERAGES:
Accuracy: 4.4
Relevance: 4.4
Safety: 5.0
Hallucination: 1.0


## Edge Case & Error Testing

In [9]:
edge_cases = [
    "",
    "   ",
    "😂😂😂😂",
    "???",
    "你好",      # unsupported
    "afhdsiofhdsiofhsd",
    "This is a long text " * 50,
]

rows = []
for text in edge_cases:
    print("=" * 70)
    print("Input:", repr(text))
    
    try:
        ans, extras = answer_user_query(text)
        err = None
    except Exception as e:
        ans, extras, err = None, None, str(e)
    
    rows.append({
        "Input": repr(text),
        "Answer": ans,
        "Extras": extras,
        "Error": err
    })

df_edge = pd.DataFrame(rows)
display(df_edge)

Input: ''
Input: '   '
Input: '😂😂😂😂'
Input: '???'
Input: '你好'
Input: 'afhdsiofhdsiofhsd'
Input: 'This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text This is a long text Thi

,Input,Answer,Extras,Error
0,'',It seems like you've provided a list of medica...,"{'detected_lang_code': 'en', 'detected_lang_na...",None
1,' ',It seems like you've provided a list of medica...,"{'detected_lang_code': 'en', 'detected_lang_na...",None
2,'😂😂😂😂',It seems like you've shared some emojis. If yo...,"{'detected_lang_code': 'en', 'detected_lang_na...",None
3,'???',It seems like your question is unclear. Could ...,"{'detected_lang_code': 'en', 'detected_lang_na...",None
4,'你好',我只是一个医疗聊天机器人，但我运作良好😊。今天我如何帮助您解决与健康相关的问题？,"{'detected_lang_code': 'zh-CN', 'detected_lang...",None
5,'afhdsiofhdsiofhsd',"Sorry, I currently support only these language...","{'detected_lang_code': 'da', 'detected_lang_na...",None
6,'This is a long text This is a long text This ...,"I'm a medical chatbot, and I can only provide ...","{'detected_lang_code': 'en', 'detected_lang_na...",None


## Performance / Latency Measurement

In [10]:
def measure_time(text, runs=3):
    times = []
    for _ in range(runs):
        start = time.time()
        ans, extras = answer_user_query(text)
        end = time.time()
        times.append(end - start)
    return np.mean(times), np.std(times)

latency_tests = [
    "I have headache",
    "எனக்கு காய்ச்சல் இருக்கு",
    "मुझे बुखार है",
]

lat_rows = []
for text in latency_tests:
    m, s = measure_time(text)
    lat_rows.append({"Input": text, "Mean (s)": m, "Std (s)": s})

df_lat = pd.DataFrame(lat_rows)
display(df_lat)

print("Overall Avg Latency:", df_lat["Mean (s)"].mean(), "seconds")

,Input,Mean (s),Std (s)
0,I have headache,4.732414,2.346784
1,எனக்கு காய்ச்சல் இருக்கு,6.714843,0.909066
2,मुझे बुखार है,13.060538,10.096144


Overall Avg Latency: 8.169265084796482 seconds
